# 🚀 Huấn Luyện Connect 4 RL (8x9) Trên Google Colab

Notebook này được **thiết kế độc lập 100% (Self-contained)**:
- **Không phụ thuộc vào git clone** hay thiếu file `connect4.py`.
- Môi trường bàn cờ 8x9 và thuật toán huấn luyện Curriculum PPO (16 agents) được tích hợp sẵn trọn vẹn.
- Tự động lưu checkpoint vào **Google Drive** và hỗ trợ **tải file zip về máy tính** chỉ với 1 click!

--- 
## 1️⃣ Bước 1: Cài đặt thư viện cần thiết

In [ ]:
!pip install -q gymnasium stable-baselines3 pygame tqdm tensorboard torch

--- 
## 2️⃣ Bước 2: Kết nối Google Drive để lưu Checkpoints
Mỗi agent sau khi train xong sẽ được lưu ngay vào Drive của bạn để chống mất dữ liệu khi ngắt kết nối.

In [ ]:
from google.colab import drive
import os

# Mount Drive
drive.mount('/content/drive')

# Thư mục lưu mô hình trên Google Drive
SAVE_DRIVE_DIR = '/content/drive/MyDrive/Connect4_Models/saved_agents_8x9'
os.makedirs(SAVE_DRIVE_DIR, exist_ok=True)

# Chế độ headless cho pygame trên máy ảo Colab
os.environ["SDL_VIDEODRIVER"] = "dummy"
print(f"✓ Môi trường sẵn sàng! Thư mục lưu Drive: {SAVE_DRIVE_DIR}")

--- 
## 3️⃣ Bước 3: Định Nghĩa Môi Trường Game Connect 4 (8 hàng x 9 cột)
*(Tự động tích hợp sẵn, không lo thiếu file `connect4.py`)*

In [ ]:
import numpy as np
import gymnasium as gym
import pygame
import time

WIN_REWARD = 10
BLOCK_REWARD = 3
DRAW_REWARD = 1
CONTINUE_REWARD = 0
ILLEGAL_MOVE_PENALTY = -100

class Connect4(gym.Env):
    def __init__(self, rows=8, cols=9):
        super().__init__()
        self.rows = rows
        self.cols = cols
        self.action_space = gym.spaces.Discrete(self.cols)
        self.observation_space = gym.spaces.Box(
            low=0, high=2, shape=(self.rows, self.cols), dtype=np.int8
        )
        self.opp_agent = None
        self.opp_epsilon = 0.1
        self.grid = np.zeros((self.rows, self.cols), dtype=np.int8)
        self.screen = None
        self.agent_symb = 1
        self.opp_symb = 2

    def step(self, action):
        if not self._is_valid_action(action):
            return (
                self.grid.copy(),
                ILLEGAL_MOVE_PENALTY,
                True,
                False,
                {"is_success": False},
            )
        row = self._drop_piece(action, self.agent_symb)
        reward, done = self._check_game_over(row, action)
        if done:
            return self.grid.copy(), reward, True, False, {"is_success": (reward != 0)}

        # Nước đi đối thủ
        opponent_action = self._get_opp_action()
        opp_row = self._drop_piece(opponent_action, self.opp_symb)
        opp_reward, opp_done = self._check_game_over(opp_row, opponent_action)
        if opp_done:
            return self.grid.copy(), -opp_reward, True, False, {"is_success": False}

        return self.grid.copy(), CONTINUE_REWARD, False, False, {}

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self.grid = np.zeros((self.rows, self.cols), dtype=np.int8)
        self.agent_symb, self.opp_symb = 1, 2
        if np.random.random() < 0.5:
            self.agent_symb, self.opp_symb = 2, 1
            action = self._get_opp_action()
            self._drop_piece(action, self.opp_symb)
        return self.grid.copy(), {"symbols": (self.agent_symb, self.opp_symb)}

    def _drop_piece(self, action, player):
        for row in range(self.rows - 1, -1, -1):
            if self.grid[row, action] == 0:
                self.grid[row, action] = player
                return row
        return None

    def _is_valid_action(self, action):
        if action < 0 or action >= self.cols:
            return False
        return self.grid[0, action] == 0

    def _get_random_action(self):
        valid_cols = [c for c in range(self.cols) if self._is_valid_action(c)]
        return np.random.choice(valid_cols) if valid_cols else 0

    def _get_opp_action(self):
        if not self.opp_agent or np.random.random() < self.opp_epsilon:
            return self._get_random_action()
        try:
            action = self.opp_agent.predict(self.grid, deterministic=False)[0]
            if isinstance(action, (np.ndarray, list, tuple)):
                action = int(action[0])
            else:
                action = int(action)
            if self._is_valid_action(action):
                return action
        except Exception:
            pass
        return self._get_random_action()

    def _check_game_over(self, row, col):
        if row is None or col is None:
            return CONTINUE_REWARD, False
        if np.all(self.grid != 0):
            return DRAW_REWARD, True

        player_symb = self.grid[row, col]
        # Ngang
        for i in range(4):
            c = col - i
            if 0 <= c and c + 3 < self.cols:
                if np.all(self.grid[row, c:c + 4] == player_symb):
                    return WIN_REWARD, True
        # Dọc
        for i in range(4):
            r = row - i
            if 0 <= r and r + 3 < self.rows:
                if np.all(self.grid[r:r + 4, col] == player_symb):
                    return WIN_REWARD, True
        # Chéo \
        for i in range(4):
            r, c = row - i, col - i
            if 0 <= r and r + 3 < self.rows and 0 <= c and c + 3 < self.cols:
                if np.all(self.grid[r:r + 4, c:c + 4].diagonal() == player_symb):
                    return WIN_REWARD, True
        # Chéo /
        for i in range(4):
            r, c = row + i, col - i
            if r < self.rows and r - 3 >= 0 and 0 <= c and c + 3 < self.cols:
                if np.all(np.fliplr(self.grid[r - 3:r + 1, c:c + 4]).diagonal() == player_symb):
                    return WIN_REWARD, True
        return CONTINUE_REWARD, False

    def set_opp_agent(self, agent, epsilon=0.1):
        self.opp_agent = agent
        self.opp_epsilon = epsilon

print("✓ Môi trường Connect4 (8x9) đã được định nghĩa thành công!")

--- 
## 4️⃣ Bước 4: Huấn Luyện Curriculum Learning Cho Bàn 8x9
Huấn luyện chuỗi 16 agents: Agent sau học cách khắc chế Agent trước đó.
Mỗi agent hoàn thành sẽ được lưu **ngay lập tức** vào Google Drive.

In [ ]:
import os
from stable_baselines3 import PPO

def train_curriculum_8x9(
    total_timesteps_per_agent=100_000,
    num_agents=16,
    save_dir='/content/drive/MyDrive/Connect4_Models/saved_agents_8x9'
):
    os.makedirs(save_dir, exist_ok=True)
    tb_log_dir = "./logs/curriculum_8x9"
    os.makedirs(tb_log_dir, exist_ok=True)

    env = Connect4(rows=8, cols=9)
    env.reset()

    print(f"Thư mục lưu checkpoints: {save_dir}")
    print(f"Bắt đầu Curriculum Learning cho bàn cờ 8x9 ({num_agents} agents)...\n")

    # --- Agent 0: Đối đầu Random ---
    agent0_path = os.path.join(save_dir, "agent0.zip")
    if os.path.exists(agent0_path):
        print("[✓] Đã tồn tại agent0 trên Google Drive, bỏ qua...")
    else:
        print("=== [8x9] Huấn luyện agent0 (đối đầu random) ===")
        agent0 = PPO("MlpPolicy", env, verbose=1, tensorboard_log=tb_log_dir)
        agent0.learn(total_timesteps=total_timesteps_per_agent, progress_bar=True, tb_log_name="PPO_agent0_8x9")
        agent0.save(os.path.join(save_dir, "agent0"))
        print("===> Hoàn tất và đã lưu agent0 vào Google Drive!\n")

    # --- Agent 1 -> num_agents-1: Đối đầu Agent trước đó ---
    for i in range(1, num_agents):
        current_agent_path = os.path.join(save_dir, f"agent{i}.zip")
        if os.path.exists(current_agent_path):
            print(f"[✓] Đã tồn tại agent{i} trên Google Drive, bỏ qua...")
            continue

        print(f"=== [8x9] Huấn luyện agent{i} (đối đầu agent{i-1}) ===")
        prev_agent_path = os.path.join(save_dir, f"agent{i-1}")
        agent_prev = PPO.load(prev_agent_path)

        agent_i = PPO("MlpPolicy", env, verbose=1, tensorboard_log=tb_log_dir)
        vec_env = agent_i.get_env()
        vec_env.env_method("set_opp_agent", agent_prev, 0.2)
        agent_i.learn(
            total_timesteps=total_timesteps_per_agent,
            progress_bar=True,
            tb_log_name=f"PPO_agent{i}_8x9",
        )
        agent_i.save(os.path.join(save_dir, f"agent{i}"))
        print(f"===> Hoàn tất và đã lưu agent{i} vào Google Drive!\n")

    print("\n🎉 TẤT CẢ AGENT ĐÃ HOÀN TẤT HUẤN LUYỆN!")

# Khởi động quá trình train (16 agents, mỗi agent 100k timesteps):
train_curriculum_8x9(total_timesteps_per_agent=100_000, num_agents=16)

--- 
## 5️⃣ Bước 5: Nén và Tải về Laptop
Chạy ô này để tải file `saved_agents_8x9.zip` về máy tính của bạn.

Sau khi tải về, bạn chỉ việc:
1. Giải nén vào thư mục `drmkn_connect4_source/drmkn_connect4_src/saved_agents_8x9/` trên laptop.
2. Chạy `python play.py` trên laptop để mở game Pygame đồ họa mượt mà!

In [ ]:
from google.colab import files
import shutil

src_dir = '/content/drive/MyDrive/Connect4_Models/saved_agents_8x9'
zip_name = '/content/saved_agents_8x9'

print("Đang nén file mô hình...")
shutil.make_archive(zip_name, 'zip', src_dir)
print("✓ Nén thành công! Bắt đầu tải xuống máy tính...")
files.download(f"{zip_name}.zip")